# COMP9444 —— LLaVA 棋子领域微调
## 第一步：官方 LLaVA-v1.5-13B 零样本 Baseline

> 严格按 `experiment_design.md`：官方原版 13B，不训练，在冻结的 **test 12 张** 上跑
> 统一评测问题，贪心解码，`extract_chess_pairs` 配对级 P/R/F1 + 整图全对率。
> 这是 LoRA / QLoRA 都要对比的**共同基准线**。

### 0. 检查 GPU（13B 4-bit 推理约需 ~9GB，T4 可跑）

In [ ]:
!nvidia-smi

### 1. 安装官方 LLaVA 仓库

不用 HF 的 `LlavaForConditionalGeneration`，改用官方 `haotian-liu/LLaVA` 仓库
（`load_pretrained_model` + 官方对话模板），和 `experiment_design.md` 保持一致。

In [ ]:
# 克隆官方仓库并以可编辑模式安装
# Clone the official LLaVA repo and install in editable mode
!git clone https://github.com/haotian-liu/LLaVA.git
%cd LLaVA
!pip install -q -e .

# 4-bit 推理需要 bitsandbytes / accelerate（CUDA 专用，Colab GPU 上可用）
# 4-bit inference needs bitsandbytes / accelerate (CUDA only, fine on Colab GPU)
!pip install -q -U bitsandbytes accelerate

In [ ]:
import torch, transformers
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

### 2. 挂载 Google Drive 并自动定位项目文件夹

数据集用共享云盘里已经传好的 `COMP9444_LLaVA_Project/data/dataset/`，不再从 HuggingFace 下载。

**给组员的说明**：直接从共享云盘里打开这个 notebook，按顺序跑即可。下面这段会
挂载你自己的 Google Drive，然后**自动**在「我的云端硬盘」和所有「共享云端硬盘」里
找 `COMP9444_LLaVA_Project`——不用手改路径，也不用加快捷方式（前提：你是该共享云盘的成员）。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

PROJECT_NAME = "COMP9444_LLaVA_Project"

# 在 MyDrive / 各共享云盘 里搜项目文件夹（共享云盘挂在 Shareddrives 或 "Shared drives"）
# Search MyDrive and every Shared Drive for the project folder
candidates = []
for base in ["/content/drive/MyDrive",
             "/content/drive/Shareddrives",
             "/content/drive/Shared drives"]:
    candidates += glob.glob(f"{base}/{PROJECT_NAME}")       # MyDrive 直接放
    candidates += glob.glob(f"{base}/*/{PROJECT_NAME}")     # 共享云盘: <盘名>/项目

# 只保留真正含 test.json 的那个，避免找到空壳
valid = [Path(c) for c in candidates
         if (Path(c) / "data" / "dataset" / "test" / "test.json").exists()]

assert valid, (
    "没找到项目文件夹。检查：\n"
    "① Drive 已挂载 ② 你是该共享云盘的成员 "
    "③ 文件夹名叫 COMP9444_LLaVA_Project 且含 data/dataset/test/test.json\n"
    f"当前扫到的候选：{candidates}"
)

PROJECT_DIR = valid[0]
DATASET_DIR = PROJECT_DIR / "data" / "dataset"
RESULT_DIR  = PROJECT_DIR / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
test_json_path = DATASET_DIR / "test" / "test.json"

print("Found project:", PROJECT_DIR)
print("Dataset:", DATASET_DIR)
print("Results:", RESULT_DIR)

### 3. 从 Drive 读取冻结的 test 集

直接读 `test/test.json`（`make_split.py` 生成，test 12 张已冻结）。
标准答案 = 每条记录的 caption。

> 注意：`test.json` 里 human 那句问题是 `make_split.py` 写的版本（不带 “in one sentence”）。
> 评测时我们**统一用下面的 `EVAL_QUESTION`**（`experiment_design.md` 的那一句），
> 只从 json 里取 caption 当标准答案 —— 这样 baseline 和微调后问的完全是同一句。

In [ ]:
import json

# 统一评测问题（baseline / LoRA / QLoRA 三边完全相同）
# The single evaluation question, identical across baseline / LoRA / QLoRA
EVAL_QUESTION = ("What chess pieces are visible in this image? "
                 "Describe their colours and piece types in one sentence.")

with open(test_json_path, encoding="utf-8") as f:
    test_json = json.load(f)

test_records = []
for r in test_json:
    test_records.append({
        "id": r["id"],
        # r["image"] 形如 "test/002.jpg"，相对 dataset/ 根
        "image_path": str(DATASET_DIR / r["image"]),
        "reference": r["conversations"][1]["value"],   # caption = 标准答案
    })

assert len(test_records) == 12, len(test_records)
print("Test set:", len(test_records), "images")
for r in test_records:
    assert Path(r["image_path"]).exists(), r["image_path"]
    print(" ", r["id"], "->", r["reference"])

### 5. 加载官方 LLaVA-v1.5-13B

`load_4bit=True` 让 13B 塞进 T4（~9GB）。如果你用的是 A100 / L4（24GB+），
可以把它改成 `load_4bit=False` 跑 fp16 全精度 baseline。

In [ ]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path

MODEL_PATH = "liuhaotian/llava-v1.5-13b"
model_name = get_model_name_from_path(MODEL_PATH)

print("Loading", MODEL_PATH, "...")
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=MODEL_PATH,
    model_base=None,
    model_name=model_name,
    load_4bit=True,      # T4 安全；A100/L4 可改 False 跑 fp16
)
model.eval()
print("Loaded. context_len =", context_len)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

### 6. Baseline 推理（贪心解码，官方对话模板）

`do_sample=False, num_beams=1` = 纯贪心，保证结果可复现。
用官方 `llava_v1` 模板拼 prompt，和微调/复测时的写法完全一致。

In [ ]:
from PIL import Image
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates
from llava.mm_utils import tokenizer_image_token, process_images

@torch.inference_mode()
def llava_answer(image, question):
    # 拼官方对话模板：USER: <image>\n{Q}  ASSISTANT:
    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(
        prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
    ).unsqueeze(0).cuda()

    image_tensor = process_images([image], image_processor, model.config)[0]

    output_ids = model.generate(
        input_ids,
        images=image_tensor.unsqueeze(0).half().cuda(),
        image_sizes=[image.size],
        do_sample=False,      # 贪心
        num_beams=1,
        max_new_tokens=128,
        use_cache=True,
    )
    return tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

# 快速自测一张
demo = Image.open(test_records[0]["image_path"]).convert("RGB")
print("Q:", EVAL_QUESTION)
print("A:", llava_answer(demo, EVAL_QUESTION))

### 7. 跑完整 test 12 张 → `baseline_results.json`

In [ ]:
import json

baseline_results = []
for idx, record in enumerate(test_records):
    image = Image.open(record["image_path"]).convert("RGB")
    reference = record["reference"]
    prediction = llava_answer(image, EVAL_QUESTION)
    baseline_results.append({
        "id": record["id"],
        "reference": reference,
        "baseline_prediction": prediction,
    })
    print(f"[{idx+1:2d}/12] {record['id']} | ref: {reference}")
    print(f"         pred: {prediction}\n")

with open(RESULT_DIR / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2, ensure_ascii=False)
print("Saved:", RESULT_DIR / "baseline_results.json")

### 8. 评分：配对级 P/R/F1 + 整图全对率

`extract_chess_pairs` 把回答解析成 `〈颜色+棋子〉` 集合，和标准答案集合比对。
比"含关键词就算对"严谨，能抓出"颜色认错"。整图全对率 = 12 张里预测集合与标准
集合**完全相等**的比例。

In [ ]:
import re, json, pandas as pd

COLORS = r"(?:white|black|brown)"
PIECES = r"(?:pawn|knight|bishop|rook|queen|king)"

def extract_chess_pairs(text):
    text = text.lower()
    pairs = set()
    for c, p in re.findall(rf"\b({COLORS})\s+(?:chess\s+)?({PIECES})\b", text):
        pairs.add(f"{c} {p}")
    for c, p in re.findall(
        rf"\b({COLORS})\s+(?:chess\s+)?piece\s+(?:being\s+)?(?:a|an)\s+({PIECES})\b",
        text):
        pairs.add(f"{c} {p}")
    return pairs

rows, tp_all, fp_all, fn_all, exact = [], 0, 0, 0, 0
for r in baseline_results:
    gold = extract_chess_pairs(r["reference"])
    pred = extract_chess_pairs(r["baseline_prediction"])
    tp, fp, fn = len(gold & pred), len(pred - gold), len(gold - pred)
    tp_all += tp; fp_all += fp; fn_all += fn
    if gold == pred:
        exact += 1
    rows.append({
        "id": r["id"],
        "gold": sorted(gold), "pred": sorted(pred),
        "tp": tp, "fp": fp, "fn": fn,
    })

prec = tp_all / (tp_all + fp_all) if tp_all + fp_all else 0.0
rec  = tp_all / (tp_all + fn_all) if tp_all + fn_all else 0.0
f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
exact_rate = exact / len(baseline_results)

print(f"Micro Precision : {prec:.3f}")
print(f"Micro Recall    : {rec:.3f}")
print(f"Micro F1        : {f1:.3f}")
print(f"整图全对率        : {exact_rate:.3f}  ({exact}/{len(baseline_results)})")

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

### 9. 保存 `baseline_metrics.json`（共同基准线）

In [ ]:
summary = {
    "model": MODEL_PATH,
    "num_test_images": len(baseline_results),
    "true_positives": tp_all,
    "false_positives": fp_all,
    "false_negatives": fn_all,
    "micro_precision": prec,
    "micro_recall": rec,
    "micro_f1": f1,
    "exact_match_rate": exact_rate,
    "per_sample": rows,
}
with open(RESULT_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("Saved:", RESULT_DIR / "baseline_metrics.json")
print("\n这就是 Baseline(13B) 那一行。LoRA / QLoRA 之后都跟它比。")